# 04 · Мастер-датасет

Единая точка входа для EDA и модели. Ноутбук собирает признаки из всех этапов в одну таблицу кандидатов. Объекты 2ГИС и районы хранятся в отдельных справочниках. Код сборки — в `notebooks/master_dataset.py`.

**Почему не одна «плоская» таблица на всё.** У данных разная гранулярность:
- кандидат (2 822 точки);
- объект 2ГИС (тысячи заведений);
- район (8).

Если склеить их в одну таблицу, строки размножатся или данные потеряются. Поэтому таблиц три, и связаны они ключами:

| таблица | одна строка = | ключ | откуда |
|---|---|---|---|
| `candidates_master.parquet` | кандидат | `candidate_id` | 02 (сетка и базовые признаки), 03_osm, 03_2gis_all_coffee, 03_2gis_demand_layers, районы |
| `poi_2gis.parquet` | объект 2ГИС | `id` (текст) | сырые ответы всех сборщиков 2ГИС; полный JSON в `raw_json` |
| `districts.parquet` | район | `district` | границы районов + statgov |

**Гарантии без потерь.**
1. Каждый источник присоединяется к сетке один к одному. Проверяется, что id уникальны и что нет ни лишних, ни потерянных кандидатов. x/y источника должны совпадать с геометрией сетки до 1 см; источник без `candidate_id` сопоставляется по координатам.
2. Если два источника дают столбец с одним именем, сборка останавливается с ошибкой: тихих переименований нет.
3. После сборки каждый столбец каждого источника сравнивается с исходным: значения, пропуски, типы.
4. Технические столбцы в таблицу кандидатов не переносятся, но остаются в справочниках. Каждый такой случай записан в `manifest.json`.
5. В `manifest.json` есть sha256 и число строк каждого входа и выхода. По нему видно, на каких данных собран датасет.

**Чего пока не хватает на этом компьютере.**
- Жильё, школы и границы районов собираются сами в `03_residential_schools_districts.ipynb`.
- Нет `almaty_features.parquet` из 02: его подготовленные слои OSM есть только у Давида.
- Нет `almaty_district_stats.csv` из statgov: нужны Excel-файлы stat.gov.kz.

Без них сборка проходит, а их отсутствие записывается в отчёт. Когда файлы появятся в `data/processed/`, достаточно перезапустить ноутбук.

In [1]:
import json
from pathlib import Path
import sys

import geopandas as gpd
import numpy as np
import pandas as pd

PROJECT_ROOT = (Path("..") if Path.cwd().name == "notebooks" else Path(".")).resolve()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import master_dataset as md_
from master_dataset import (FEATURE_SOURCES, FeatureSource, MasterBuildError, assert_lossless, assign_districts,
                            build_district_table, build_master, build_poi_2gis, check_split_copies, load_master)

pd.set_option("display.max_columns", 80)
for s in FEATURE_SOURCES:
    print(f"{s.name:12s} {'есть' if (PROJECT_ROOT / s.path).is_file() else 'НЕТ '}  {s.path}  ← {s.owner}")

base02       НЕТ   data/processed/almaty_features.parquet  ← 02_candidate_locations
osm          есть  data/processed/features/osm_commercial_features.csv  ← 03_osm_commercial_features
osm_context  есть  data/processed/features/osm_residential_schools.csv  ← 03_residential_schools_districts
dgis_coffee  есть  data/processed/features/dgis_coffee_features.csv  ← 03_2gis_all_coffee
dgis_demand  есть  data/processed/features/dgis_demand_features.csv  ← 03_2gis_demand_layers


## Офлайн-тесты сборки

Тесты собирают маленький искусственный проект и проверяют:
- что сохраняются значения и типы, включая пропуски, `Int64` и `bool`;
- что ловятся расхождения x/y, лишние и потерянные id, одинаковые имена столбцов;
- назначение района внутри полигона, на границе и по ближайшему, с разным написанием названий;
- справочник объектов 2ГИС из сырых ответов;
- что отсутствующий источник попадает в отчёт.

In [2]:
import sqlite3
import tempfile
import unittest
from contextlib import closing

import shapely

import dgis_api


def make_project(tmp, n=6):
    root = Path(tmp)
    (root / "data/processed/features").mkdir(parents=True)
    xs = 650_000.0 + 500 * np.arange(n)
    grid = gpd.GeoDataFrame({"candidate_id": np.arange(n, dtype="int64")},
                            geometry=gpd.points_from_xy(xs, np.full(n, 4_785_000.0)), crs="EPSG:32643")
    grid.to_file(root / "data/processed/candidate_locations.gpkg", driver="GPKG")
    boundary = gpd.GeoDataFrame(geometry=[shapely.box(xs[0] - 100, 4_784_900, xs[-1] + 100, 4_785_100)], crs="EPSG:32643")
    boundary.to_file(root / "data/processed/almaty_boundary.gpkg", driver="GPKG")
    feats = pd.DataFrame({"candidate_id": grid["candidate_id"], "x": grid.geometry.x, "y": grid.geometry.y,
                          "shops_500m": [3, 0, 1, 2, 5, 0], "metro_dist": [100.5, np.nan, 300.0, 1.0, 2.0, 3.0],
                          "flag": [True, False, True, True, False, True]}).sample(frac=1, random_state=1)
    feats.to_csv(root / "data/processed/features/a.csv", index=False)
    geo = grid.sample(frac=1, random_state=2).drop(columns="candidate_id").reset_index(drop=True)
    geo["cafes_500m"] = [7, 1, 0, 2, 4, 3]
    geo.to_parquet(root / "data/processed/base.parquet")
    return root, grid


SOURCES = (FeatureSource("a", "data/processed/features/a.csv", "test_a"),
           FeatureSource("base", "data/processed/base.parquet", "test_base", key="geometry"))


class MasterTests(unittest.TestCase):
    def setUp(self):
        temp = tempfile.TemporaryDirectory()
        self.addCleanup(temp.cleanup)
        self.root, self.grid = make_project(temp.name)

    def build(self, sources=SOURCES, **kw):
        return build_master(self.root, sources=sources, expected=6, with_poi=False, **kw)

    def test_lossless_roundtrip_and_geometry_join(self):
        result = self.build()
        master = result["master"]
        self.assertEqual(len(master), 6)
        a = pd.read_csv(self.root / "data/processed/features/a.csv").set_index("candidate_id").sort_index()
        pd.testing.assert_series_equal(master.set_index("candidate_id")["metro_dist"], a["metro_dist"], check_names=False)
        geo = gpd.read_parquet(self.root / "data/processed/base.parquet")
        by_x = dict(zip(geo.geometry.x.round(3), geo["cafes_500m"]))
        self.assertEqual(master["cafes_500m"].tolist(), [by_x[round(x, 3)] for x in master["x"]])
        back = gpd.read_parquet(self.root / "data/processed/master/candidates_master.parquet")
        pd.testing.assert_frame_equal(pd.DataFrame(back.drop(columns="geometry")),
                                      pd.DataFrame(master.drop(columns="geometry")))
        manifest = json.loads((self.root / "data/processed/master/manifest.json").read_text(encoding="utf-8"))
        self.assertIn("district", manifest["skipped_sources"])
        self.assertEqual(manifest["outputs"]["candidates_master.parquet"]["rows"], 6)
        self.assertEqual(set(result["dictionary"]["column"]),
                         {"candidate_id", "x", "y", "lon", "lat", "shops_500m", "metro_dist", "flag", "cafes_500m"})

    def test_xy_mismatch_is_rejected(self):
        a = pd.read_csv(self.root / "data/processed/features/a.csv")
        a.loc[0, "x"] += 0.02
        a.to_csv(self.root / "data/processed/features/a.csv", index=False)
        with self.assertRaisesRegex(MasterBuildError, "x/y"):
            self.build()

    def test_missing_and_extra_ids_are_rejected(self):
        a = pd.read_csv(self.root / "data/processed/features/a.csv")
        for broken in (a.iloc[1:], pd.concat([a, a.iloc[[0]].assign(candidate_id=99)]), pd.concat([a, a.iloc[[0]]])):
            broken.to_csv(self.root / "data/processed/features/a.csv", index=False)
            with self.assertRaises(MasterBuildError):
                self.build()

    def test_column_collision_is_rejected(self):
        twin = FeatureSource("twin", "data/processed/features/a.csv", "test_twin")
        with self.assertRaisesRegex(MasterBuildError, "Одинаковые столбцы"):
            self.build(sources=SOURCES + (twin,))

    def test_changed_value_is_detected(self):
        result = self.build(write=False)
        frames = {"a": pd.read_csv(self.root / "data/processed/features/a.csv").set_index("candidate_id").drop(columns=["x", "y"])}
        broken = result["master"].copy()
        broken.loc[2, "shops_500m"] = 999
        with self.assertRaises(MasterBuildError):
            assert_lossless(broken, frames)

    def test_missing_source_is_reported(self):
        ghost = FeatureSource("ghost", "data/processed/features/ghost.csv", "test_ghost")
        result = self.build(sources=SOURCES + (ghost,), write=False)
        self.assertIn("ghost", result["skipped"])

    def test_districts_within_boundary_nearest_and_names(self):
        x0 = 650_000.0
        districts = gpd.GeoDataFrame({"NAME": ["Ауэзовский ", "Медеуский"], "OBJECTID": [1, 2], "RAION": ["312", "315"]},
                                     geometry=[shapely.box(x0 - 250, 4_784_000, x0 + 1000, 4_786_000),
                                               shapely.box(x0 + 1000, 4_784_000, x0 + 1600, 4_786_000)], crs="EPSG:32643")
        stats = pd.DataFrame({"district": ["Ауезовский район", "Медеуский"], "population_2026": [362597, 257104]})
        assigned = assign_districts(self.grid, districts)
        self.assertEqual(assigned["district"].tolist(),
                         ["Ауэзовский", "Ауэзовский", "Ауэзовский", "Медеуский", "Медеуский", "Медеуский"])
        self.assertEqual(assigned["district_method"].tolist(), ["within", "within", "boundary", "within", "nearest", "nearest"])
        self.assertEqual(assigned["district_distance_m"].round(1).tolist(), [0, 0, 0, 0, 400.0, 900.0])
        dim = build_district_table(districts, stats)
        self.assertEqual(dim["population_2026"].tolist(), [362597, 257104])
        with self.assertRaisesRegex(MasterBuildError, "не совпали"):
            build_district_table(districts, stats.assign(district=["Алмалинский", "Медеуский"]))
        districts.to_parquet(self.root / "data/processed/almaty_districts.parquet")
        stats.to_csv(self.root / "data/processed/almaty_district_stats.csv", index=False)
        master = self.build(write=False)["master"]
        self.assertEqual(master["district_population_2026"].tolist(), [362597] * 3 + [257104] * 3)
        self.assertNotIn("district_OBJECTID", master.columns)
        self.assertIn("district_RAION", master.columns)
        # без статистики: районы и атрибуты границ подключаются, отсутствие статистики — в отчёте
        (self.root / "data/processed/almaty_district_stats.csv").unlink()
        only_bounds = self.build(write=False)
        self.assertIn("district_stats", only_bounds["skipped"])
        self.assertEqual(only_bounds["master"]["district_RAION"].tolist(), ["312"] * 3 + ["315"] * 3)
        self.assertNotIn("district_population_2026", only_bounds["master"].columns)

    def test_poi_from_raw_checkpoints(self):
        coffee = self.root / dgis_api_paths["coffee"]
        coffee.parent.mkdir(parents=True)
        with closing(sqlite3.connect(coffee)) as db:
            db.executescript(dgis_api.AREA_SCHEMA)
            db.execute("INSERT INTO run (id, plan) VALUES (1, '{}')")
            items = [{"id": "701", "name": "Кофейня 1", "point": {"lon": 76.9, "lat": 43.2},
                      "rubrics": [{"id": "162", "name": "Кофейни", "kind": "primary"}],
                      "attribute_groups": [{"attributes": [{"tag": "food_service_avg_price", "name": "Чек 2500 ₸"}]}]},
                     {"id": "702_hash", "name": "Точка", "point": {"lon": 76.91, "lat": 43.2},
                      "rubrics": [{"id": "58871", "name": "Точки кофе", "kind": "primary"}]}]
            db.executemany("INSERT INTO items VALUES (?, ?)", [(dgis_api.base_id(i["id"]), json.dumps(i)) for i in items])
            db.commit()
        gap = self.root / dgis_api_search["gap"]
        gap.parent.mkdir(parents=True)
        with closing(sqlite3.connect(gap)) as db:
            db.executescript(dgis_api.SCHEMA)
            db.execute("INSERT INTO run (id, plan) VALUES (1, '{}')")
            db.executemany("INSERT INTO organizations VALUES (?, ?, ?, ?, ?, ?)",
                           [("701", "Кофейня 1", 76.9, 43.2, 4.5, 10), ("703", "Кафе из поиска", 76.92, 43.2, 4.0, 3)])
            db.commit()
        poi, used = build_poi_2gis(self.root, area=shapely.box(0, 0, 1e7, 1e7))
        self.assertEqual(poi["id"].tolist(), ["701", "702", "703"])
        self.assertEqual(poi["sources"].tolist(), ["coffee;gap", "coffee", "gap"])
        self.assertEqual(poi["is_coffee_shop"].tolist(), [True, False, False])
        self.assertEqual(poi["is_coffee_point"].tolist(), [False, True, False])
        self.assertEqual(poi.loc[0, "avg_check_kzt"], 2500)
        self.assertTrue(pd.isna(poi.loc[2, "raw_json"]))   # объект только из текстового поиска — JSON нет
        self.assertEqual(json.loads(poi.loc[1, "raw_json"])["id"], "702_hash")
        poi.to_parquet(self.root / "poi.parquet")
        back = gpd.read_parquet(self.root / "poi.parquet")
        self.assertEqual(back["id"].tolist(), poi["id"].tolist())


dgis_api_paths, dgis_api_search = md_.DGIS_AREA_CHECKPOINTS, md_.DGIS_SEARCH_CHECKPOINTS
suite = unittest.defaultTestLoader.loadTestsFromTestCase(MasterTests)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError("Проверки сборки мастер-датасета завершились ошибкой.")

test_changed_value_is_detected (__main__.MasterTests.test_changed_value_is_detected) ... 

ok


test_column_collision_is_rejected (__main__.MasterTests.test_column_collision_is_rejected) ... 

ok


test_districts_within_boundary_nearest_and_names (__main__.MasterTests.test_districts_within_boundary_nearest_and_names) ... 

ok


... (7 outputs left out)

ok


test_xy_mismatch_is_rejected (__main__.MasterTests.test_xy_mismatch_is_rejected) ... 

ok


----------------------------------------------------------------------
Ran 8 tests in 1.656s

OK


## Сборка

Отдельные CSV признаков OSM (`shops_500m.csv` и т. п.) должны совпадать со сводным `osm_commercial_features.csv`. Это проверяется перед сборкой. В мастер-таблицу входит только сводный файл, чтобы не было двух копий одних данных.

In [3]:
print("Отдельные файлы OSM совпадают со сводным:", check_split_copies(PROJECT_ROOT, "data/processed/features/osm_commercial_features.csv"))
result = build_master(PROJECT_ROOT)
master, dictionary, poi = result["master"], result["dictionary"], result["poi"]
print(f"\nМастер-таблица: {master.shape[0]} кандидатов × {master.shape[1]} столбцов")
for r in result["reports"]:
    print(f"  {r['name']:12s} ключ={r['key']:12s} строк={r['rows']}  признаков={len(r['columns'])}  "
          f"x/y до {r.get('xy_max_diff_m', 0):.4f} м  не перенесено: {list(r['dropped'])}")
print("Пропущенные источники:")
for name, reason in result["skipped"].items():
    print(f"  {name}: {reason}")
print(f"\nОбъекты 2ГИС: {len(poi)} (внутри города + 1 км: {int(poi['in_area'].sum())}); источники: {result['poi_sources']}")
print("Выходы:", json.dumps(result["outputs"], ensure_ascii=False, indent=1))

Отдельные файлы OSM совпадают со сводным: ['attractions_1000m.csv', 'major_roads_distance.csv', 'malls_1000m.csv', 'metro_dist.csv', 'parking_500m.csv', 'restaurants_500m.csv', 'shops_500m.csv', 'transit_stops_500m.csv']



Мастер-таблица: 2822 кандидатов × 53 столбцов
  osm          ключ=candidate_id строк=2822  признаков=8  x/y до 0.0000 м  не перенесено: ['x', 'y']
  osm_context  ключ=candidate_id строк=2822  признаков=5  x/y до 0.0000 м  не перенесено: ['x', 'y']
  dgis_coffee  ключ=candidate_id строк=2822  признаков=8  x/y до 0.0000 м  не перенесено: ['x', 'y']
  dgis_demand  ключ=candidate_id строк=2822  признаков=19  x/y до 0.0000 м  не перенесено: ['x', 'y']
Пропущенные источники:
  base02: нет файла data/processed/almaty_features.parquet (создаёт 02_candidate_locations)
  district_stats: нет data/processed/almaty_district_stats.csv (создаёт 03_statgov_exploration из Excel stat.gov.kz)

Объекты 2ГИС: 3756 (внутри города + 1 км: 3503); источники: {'coffee': {'path': 'data/processed/2gis_coffee/coffee_progress.sqlite3', 'items': 1517}, 'offices': {'path': 'data/processed/2gis_layers/offices_progress.sqlite3', 'items': 383}, 'bakeries': {'path': 'data/processed/2gis_layers/bakeries_progress.sqlite3'

## Словарь данных и профиль

Словарь — `data/processed/master/data_dictionary.csv`. Для каждого столбца там указаны источник, ноутбук-владелец, тип, число пропусков и описание.

In [4]:
display(dictionary)
print("Столбцы с пропусками:", dictionary.loc[dictionary["null"] > 0, ["column", "null", "description"]].to_dict("records"))

,column,source,owner,dtype,non_null,null,description
0,candidate_id,base,02_candidate_locations,int64,2822,0,"ID точки-кандидата (из 02, сетка 500 м)"
1,x,base,02_candidate_locations,float64,2822,0,"Координата X, м (EPSG:32643)"
2,y,base,02_candidate_locations,float64,2822,0,"Координата Y, м (EPSG:32643)"
3,lon,base,02_candidate_locations,float64,2822,0,Долгота (WGS84)
4,lat,base,02_candidate_locations,float64,2822,0,Широта (WGS84)
5,shops_500m,osm,03_osm_commercial_features,int64,2822,0,Магазины OSM в 500 м (после склейки дублей)
6,malls_1000m,osm,03_osm_commercial_features,int64,2822,0,ТРЦ OSM в 1000 м
7,restaurants_500m,osm,03_osm_commercial_features,int64,2822,0,Рестораны/кафе/фастфуд OSM в 500 м
8,metro_dist,osm,03_osm_commercial_features,float64,2822,0,"Расстояние до станции метро, м"
9,transit_stops_500m,osm,03_osm_commercial_features,int64,2822,0,Остановки ОТ в 500 м (без дублей)


Столбцы с пропусками: [{'column': 'dgis_coffee_rating_mean_500m', 'null': 2202, 'description': 'Средний рейтинг кофеен в 500 м'}, {'column': 'dgis_coffee_avg_check_median_500m', 'null': 2293, 'description': 'Медианный средний чек кофеен в 500 м, ₸'}]


In [5]:
layer_cols = [c for c in poi.columns if c.startswith("is_")]
inside = poi[poi["in_area"]]
display(inside[layer_cols].sum().rename("объектов в городе + 1 км").to_frame())
print("Объектов сразу в нескольких слоях:", int((inside[layer_cols].sum(axis=1) > 1).sum()))

,объектов в городе + 1 км
is_coffee_shop,971
is_coffee_point,490
is_business_center,308
is_coworking,70
is_bakery,293
is_cafe_confectionery,243
is_fitness,449
is_education,268
is_mall,195
is_coffee_vending,485


Объектов сразу в нескольких слоях: 258


## Как пользоваться

```python
from master_dataset import load_master
master = load_master()                      # GeoDataFrame: 2 822 кандидата × все признаки, EPSG:32643
X = load_master(with_geometry=False)        # то же без геометрии — для модели

import geopandas as gpd
poi = gpd.read_parquet("../data/processed/master/poi_2gis.parquet")   # объекты 2ГИС, id — текст
```

Мастер-таблицу не редактируйте вручную. Новый признак добавляется так:
1. ноутбук этапа 03 пишет CSV с `candidate_id, x, y, <признаки>`;
2. в `FEATURE_SOURCES` в `master_dataset.py` добавляется одна строка;
3. ноутбук 04 перезапускается.